In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

import numpy as np

from pandas import ExcelWriter

from time import sleep

import os

import re

import tabula

from tabula.io import read_pdf

from selenium import webdriver

from selenium.webdriver.common.by import By




In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'MX CNSF' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Read {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



Running MX CNSF Web Scraping Tool v.1.2


In [3]:


# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [ ]:


# %%

#------------------------------------------------ Begin_variable----------------------------------------

regdict={

        'MX CNSF 2': "//a[contains (@onclick,'Oficinas de Representación')]", 

        'MX CNSF 3': "//a[contains (@onclick,'Registro General de Reaseguradoras Extranjeras')]"

         }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

          'Phone - Mother company': [], 'Check': []}



ISO_MX={"EspaÃ±a" :"ES", "E.U.A." :"US", "Alemania" :"DE","Bermuda" :"BM",

        "Francia" :"FR","PanamÃ¡" :"PA","Unido." :"GB", " México": "MX", " Mexico": "MX"}



columns2=["Index", "NAME", "No. de Registro",'REPRESENTANTE LEGAL','DOMICILIO']

columns3=['Index','NAME', 'No. de Registro', 'Operación']



processdate = now.strftime('%Y-%m-%d')




In [5]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def add_empty_cols(df, final_cols: int):

    rows, cols = df.shape

    if final_cols < cols:

        print(f"DataFrame has more than {final_cols} columns ({cols} columns found) returning same DataFrame")

        return df

    for i in range(final_cols-cols):

        col_name = f"new_{i}"

        df[col_name] = np.nan

    return df




In [6]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get('https://www.gob.mx/cnsf/documentos/reaseguradores-extranjeros-e-intermediarios')

for reg in regdict:

    print(f'[INFO] : Working with {reg}')



    link_cnsf= driver.find_element(By.XPATH,f"{regdict[reg]}")

    file_cnsf= link_cnsf.get_attribute('href')

    cnsf = tabula.io.read_pdf(file_cnsf,lattice=True,pages='all',pandas_options={'dtype':str,'header':None})



    for df in cnsf:

        count_df = 0



        if reg == 'MX CNSF 2' :

            df.columns = columns2[:df.shape[1]]

            df = df.fillna("")  # subsitute nan with empty strings

            df = df.reset_index(drop=True)

            df['No. de Registro'] = df['No. de Registro'].str.replace('\r', '')

            df['No. de Registro']= df['No. de Registro'].str.split('(').str[0]

            df['Email_address'] = df['DOMICILIO'].str.findall('(\S+@\S+)')

            df['Email_address'] = df['Email_address'].apply(lambda x: ','.join(map(str, x)))

            df['Phone_number']  = df['DOMICILIO'].str.findall("(\d{2}[\W]*?\d{2}[\W]*?\d{2}[\W]*?\d{2}[\W]*?\d{2})")

            df['Phone_number']  = df['Phone_number'].apply(lambda x: ','.join(map(str, x)))

            df['Phone_number']  = df['Phone_number'].apply(lambda row:row.replace('\r',''))

            df['Fax']           = df['DOMICILIO'].str.extract(r"(\W*?fax\W*(\d{2}[-]?)+)")[0]

            df['Fax']           = df['Fax'].str.split(':').str[1]

            address             = df['DOMICILIO'].str.split('C.P.')  

            df['Address']       = address.str[0].replace('C.P.','')

            address             = df['Address'].str.split('Código')  

            df['Address']       = address.str[0].replace('Código','')

            df['PostCode']      = df['DOMICILIO'].str.extract('(\d{5})')

            

        else:

            df.columns = columns3[:df.shape[1]]



        df = df.fillna("")  # subsitute nan with empty strings

        df = df.reset_index(drop=True)



        for index, row in df.iterrows():

            if row['Index'].isdigit() and len(row['Index'])>0:

                count_df +=1



                if reg == 'MX CNSF 2' :

                    addr_cleaning = row['Address'].strip().replace('\r', ' ') if row['Address'].strip()[-1]!=',' else row['Address'].strip()[:-1].replace('\r', ' ')

                    Phone   = row['Phone_number']

                    Fax     = row['Fax']

                    Email   = row['Email_address']

                    Zip     = row['PostCode']

                else:

                    addr_cleaning = ''

                    Phone = ''

                    Fax = ''

                    Email = ''

                    Zip = ''



                

                sqldict['Phone'].append(Phone)

                sqldict['Fax'].append(Fax)

                sqldict['Email'].append(Email)

                sqldict['Address_1'].append(addr_cleaning)

                sqldict['Zip'].append(Zip)  



                if 'No. de Registro' in list(df.columns):    

                    if len(row['No. de Registro'].split('-'))==4 and row['No. de Registro'].split('-')[0]=='RGRE' :

                        InternalID_1 = row['No. de Registro']

                        InternalID_1_type = 'No. de Registro'

                        Name = row['NAME']

                    else :

                        InternalID_1 = ''

                        InternalID_1_type = ''                        

                        Name = row['No. de Registro']    

                else :

                    InternalID_1 = ''

                    InternalID_1_type = '' 

                    Name = row['NAME']



                if Name.find('RGRE')!=-1 :

                    InternalID_1 = 'RGRE'+ Name.split('RGRE')[-1].strip()

                    InternalID_1_type = 'No. de Registro'

                    Name = Name.split('RGRE')[0].strip()



                sqldict['Name'].append(Name)

                sqldict['InternalID_1'].append(InternalID_1)

                sqldict['InternalID_1_type'].append(InternalID_1_type)

                sqldict['RegulationType'].append('Supervised')

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0])

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1])



        print(f'[INFO] : -- Len dataFrame = {count_df} | {reg}')

        sqldict = bourange_same_length_array(sqldict)

        sleep(1)





<>:39: SyntaxWarning: invalid escape sequence '\S'
<>:43: SyntaxWarning: invalid escape sequence '\d'
<>:61: SyntaxWarning: invalid escape sequence '\d'
<>:39: SyntaxWarning: invalid escape sequence '\S'
<>:43: SyntaxWarning: invalid escape sequence '\d'
<>:61: SyntaxWarning: invalid escape sequence '\d'
C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_16596\1296408704.py:39: SyntaxWarning: invalid escape sequence '\S'
  df['Email_address'] = df['DOMICILIO'].str.findall('(\S+@\S+)')
C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_16596\1296408704.py:43: SyntaxWarning: invalid escape sequence '\d'
  df['Phone_number']  = df['DOMICILIO'].str.findall("(\d{2}[\W]*?\d{2}[\W]*?\d{2}[\W]*?\d{2}[\W]*?\d{2})")
C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_16596\1296408704.py:61: SyntaxWarning: invalid escape sequence '\d'
  df['PostCode']      = df['DOMICILIO'].str.extract('(\d{5})')


[INFO] : Working with MX CNSF 3


Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'


[INFO] : -- Len dataFrame = 0 | MX CNSF 3
[INFO] : -- Len dataFrame = 79 | MX CNSF 3
[INFO] : -- Len dataFrame = 85 | MX CNSF 3
[INFO] : -- Len dataFrame = 87 | MX CNSF 3
[INFO] : -- Len dataFrame = 13 | MX CNSF 3
[INFO] : -- Len dataFrame = 6 | MX CNSF 3
[INFO] : -- Len dataFrame = 15 | MX CNSF 3
[INFO] : -- Len dataFrame = 13 | MX CNSF 3
[INFO] : -- Len dataFrame = 7 | MX CNSF 3
[INFO] : -- Len dataFrame = 21 | MX CNSF 3
[INFO] : -- Len dataFrame = 2 | MX CNSF 3
[INFO] : -- Len dataFrame = 22 | MX CNSF 3
[INFO] : -- Len dataFrame = 11 | MX CNSF 3
[INFO] : -- Len dataFrame = 13 | MX CNSF 3


In [8]:


# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

# Supprimer les lignes où la colonne 'Name' est vide

df = df[df['Name'] != '']

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    

C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_16596\1571250099.py:13: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,ACE INA OVERSEAS INSURANCE COMPANY LTD.,RGRE-1127-14-328972,No. de Registro,,,...,,,,,,,,,,
1,,,,,,ACE PROPERTY AND CASUALTY INSURANCE COMPANY,RGRE-193-85-300168,No. de Registro,,,...,,,,,,,,,,
2,,,,,,ACTIVE CAPITAL REINSURANCE LTD.,RGRE-1191-15-C0000,No. de Registro,,,...,,,,,,,,,,
3,,,,,,AFFILIATED FM INSURANCE COMPANY,RGRE-1197-16-C0000,No. de Registro,,,...,,,,,,,,,,
4,,,,,,AGCS MARINE INSURANCE COMPANY,RGRE-1040-09-328293,No. de Registro,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,,,,,,THE TOA 21ST CENTURY REINSURANCE COMPANY LTD.,,,,,...,,,,,,,,,,
370,,,,,,POOL SYNDICAT BELGE D ́ASSURANCES NUCLÉAIRES (...,RGRE-1296-23-C0000,No. de Registro,,,...,,,,,,,,,,
371,,,,,,HDI GLOBAL SE,,,,,...,,,,,,,,,,
372,,,,,,AG INSURANCE,,,,,...,,,,,,,,,,
